In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from os import mkdir, listdir
from os.path import isdir, isfile

In [ ]:
def load_unique_celltype_dataframe(unique_celltype_path:str)->pd.DataFrame:
    # loading the unique_celltypes table generated by the analyze_deseq.R script
    # this table inherits genes mapped to celltypes, the genes do not occur in 2 other celltypes
    unique_celltypes = pd.read_table(unique_celltype_path, sep=";")
    print("[*] Length of unique_celltypes table: {}".format(len(unique_celltypes)))
    return unique_celltypes

def load_log2FoldChange_dataframe(table_path:str)->pd.DataFrame:
    # loading log2FoldChange dataframes from deseq2
    differential_gene_expression_df = pd.read_csv(table_path)
    print("[*] Length of diff. expression table: {}".format(len(differential_gene_expression_df)))
    print("[*] Changing first column header of diff. expression table: {}, to {}".format("Unnamed: 0","ID"))
    differential_gene_expression_df.rename(columns={"Unnamed: 0":"ID"}, inplace=True)
    differential_gene_expression_df.ID = differential_gene_expression_df.ID.apply(lambda x: "HVAEP1-" + x.split(".")[1])
    return differential_gene_expression_df

def merging_celltype_and_lof2FoldChange_dfs(unique_celltypes:pd.DataFrame, 
                                            differential_gene_expression_df:pd.DataFrame)->pd.DataFrame:
    print("[*] merging both dataframes on ID column ...")
    merged_table = unique_celltypes.merge(differential_gene_expression_df, on="ID")
    print("[*] Length of merged table: {}".format(len(merged_table)))
    return merged_table

def filter_merged_dataframe(merged_table:pd.DataFrame)->tuple:
    # filtering the table for padj <= 0.05 and diff. expressed genes
    padj_merged_table = merged_table[merged_table.padj <= 0.05]
    print("[*] Filtered for padj <= 0.05, new length of merged table: {}".format(len(padj_merged_table)))
    # filtering for up and down regulated genes
    upregulated_genes_to_celltypes = padj_merged_table[padj_merged_table.log2FoldChange >= 1.0]
    downregulated_genes_to_celltypes = padj_merged_table[padj_merged_table.log2FoldChange <= -1.0]
    print("[*] Filtering for up and down regulated genes ...")
    print("\t[*] Amount of upregulated genes: {}".format(len(upregulated_genes_to_celltypes)))
    print("\t[*] Amount of downregulated genes: {}".format(len(downregulated_genes_to_celltypes)))
    return upregulated_genes_to_celltypes, downregulated_genes_to_celltypes

def plotting_barh_up_down_celltype_number(upregulated_genes_to_celltypes:pd.DataFrame,
                                          downregulated_genes_to_celltypes:pd.DataFrame,
                                          experiment:str, savep:str, celltype_list:list)->int:
    plot_type = "upregulated"
    image_path = savep + plot_type + "_unique_celltypes_" + experiment + ".png"
    print("[*] Creating upregulated celltype horizontal bar plot ...")
    up_summation = upregulated_genes_to_celltypes[celltype_list].fillna(0).sum()
    ax = up_summation[up_summation != 0].plot(kind="barh",
                                         figsize=(12,8),
                                         grid=True, 
                                         edgecolor="black",
                                         color="lightblue",title="Unique celltypes for upregulated genes in {}".format(experiment))
    for i in ax.patches:
        ax.text(i.get_width() + 0.1, i.get_y() + 0.1, str(round(i.get_width(), 2)), fontsize=8, color='black')
    print("\t[*] Saving file to: {}".format(image_path))

    plt.savefig(image_path, dpi=400)
    plt.close()

    plot_type = "downregulated"
    image_path = savep + plot_type + "_unique_celltypes_" + experiment + ".png"
    print("[*] Creating upregulated celltype horizontal bar plot ...")
    down_summation = downregulated_genes_to_celltypes[celltype_list].fillna(0).sum()
    ax = down_summation[down_summation != 0].plot(kind="barh",
                                         figsize=(12,8),
                                         grid=True, 
                                         edgecolor="black",
                                         color="lightcoral",title="Unique celltypes for downregulated genes in {}".format(experiment))
    for i in ax.patches:
        ax.text(i.get_width() + 0.1, i.get_y() + 0.1, str(round(i.get_width(), 2)), fontsize=8, color='black')
    
    print("\t[*] Saving file to: {}".format(image_path))
    plt.savefig(image_path, dpi=400)
    plt.close()
    print("[+] DONE")
    return 0

def get_data_dict(data_path:str)->tuple:
    print("[*] Loading experiments with corresponding dataframes ...")
    experiment_to_data_dict = {}
    for file in listdir(data_path):
        if file.endswith(".csv"):
            if "HydraRecolonization_Conventionalized" in file or "HydraRecolonization_Cvbct" in file:
                exp = file.split("_results")[0]
            else:
                exp = file.split("_vs_")[0]
            experiment_to_data_dict[exp] = data_path + file
    print("[+] DONE")
    return experiment_to_data_dict

def analyze_celltype_vs_diff_dataframes(unique_celltypes:pd.DataFrame,
                                        log2FoldChange_path:str, 
                                        savep_figures:str,
                                        savep_tables:str,
                                        celltype_list:str,
                                        exp:str)->int:
    differential_gene_expression_df = load_log2FoldChange_dataframe(table_path=log2FoldChange_path)
    merged_table = merging_celltype_and_lof2FoldChange_dfs(unique_celltypes=unique_celltypes,
                                                           differential_gene_expression_df=differential_gene_expression_df)

    upregulated_genes_to_celltypes, downregulated_genes_to_celltypes = filter_merged_dataframe(merged_table=merged_table)
    
    print("[*] Producing output tables ...")
    print("[*] Using directory: {}".format(savep_tables))
    upregulated_genes_to_celltypes.to_excel(savep_tables + "upregulated_celltypes_" + exp + ".xlsx")
    upregulated_genes_to_celltypes.to_csv(savep_tables + "upregulated_celltypes_" + exp + ".csv")
    downregulated_genes_to_celltypes.to_excel(savep_tables + "downregulated_celltypes_" + exp + ".xlsx")
    downregulated_genes_to_celltypes.to_csv(savep_tables + "downregulated_celltypes_" + exp + ".csv")
    print("[+] DONE")
    return plotting_barh_up_down_celltype_number(upregulated_genes_to_celltypes=upregulated_genes_to_celltypes,
                                               downregulated_genes_to_celltypes=downregulated_genes_to_celltypes,
                                               experiment=exp,
                                               savep=savep_figures,
                                               celltype_list=celltype_list)

In [ ]:
savep = "../results/figures/differential_expression_celltype_analysis/"
savep_tables = savep = "../results/processedData/differential_expression_celltype_analysis/"
unique_celltypes_path = "../results/processedData/normalized_mean/unique_celltypes.table"
unique_celltypes = load_unique_celltype_dataframe(unique_celltype_path=unique_celltypes_path)
celltype_list = list(unique_celltypes.columns[1:])
experiment_to_data_dict = get_data_dict("../results/deseq2_rsem/tables/")

for exp in experiment_to_data_dict.keys():
    print("[*] Starting Analysis with: {}".format(exp))
    log2FoldChange_path = experiment_to_data_dict[exp]
    analyze_celltype_vs_diff_dataframes(unique_celltypes=unique_celltypes,
                                        log2FoldChange_path=log2FoldChange_path,
                                        savep_figures=savep,
                                        savep_tables=savep_tables,
                                        celltype_list=celltype_list,
                                        exp=exp)

In [ ]:
# superclusters are super usefull in detecting "outlier" genes such as:
# HVAEP1-G011796 which is not uniquely expressed in the Ec_peduncle but in the C_EC_Foot, L_ECTODERM and C_head cluster ...

In [ ]:
for i in merged_table[merged_table.ID == "HVAEP1-G011796"]:
    if merged_table[merged_table.ID == "HVAEP1-G011796"][i].values[0] == 1.0:
        print("\t[*] HVAEP1-G011796 occurence:", i, merged_table[merged_table.ID == "HVAEP1-G011796"][i].values[0])